In [1]:
import os
import pandas as pd
import numpy as np

from cv_tools import utils
import config as C

%load_ext autoreload
%autoreload 1
%aimport config
%aimport cv_tools.utils

data_dir = C.DATA_DIR

/Users/s2785075/miniconda3/envs/cv_tools/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DeepFish

In [ ]:
dataset_dir = os.path.join(data_dir, 'resources', 'DeepFish_YOLO-Fish')
dataset = utils.yolo_to_coco(dataset_dir=dataset_dir, dataset_info='DeepFish dataset with additional annotations for training of YOLO-Fish (https://doi.org/10.1016/j.ecoinf.2022.101847 and https://doi.org/10.1038/s41598-020-71639-x)')
utils.save_json(dataset, os.path.join(dataset_dir, 'DeepFish.json'))

# S-UODAC

In [24]:
dataset_dir = os.path.join(data_dir, 'resources', 'S-UODAC')

domains = {}
for d in range(1, 8):
    domains[d] = pd.read_csv(os.path.join(dataset_dir, 'domains', f'type{d}.txt'), dtype=str, header=None)[0].tolist()

for fn, suffix in [('instances_source.json', 'train'), ('instances_target.json', 'test')]:
    dataset = utils.read_json(os.path.join(dataset_dir, 'raw', fn), assert_correct=False)
    dataset['info'] = dict(description='Synthetic Underwater Object Detection Algorithm Contest 2020 (https://doi.org/10.1016/j.neucom.2023.01.053)')
    
    for img in dataset['images']:
        new_fn = None
        for d in domains:
            if '.'.join(img['file_name'].split('.')[:-1]) in domains[d]:
                assert new_fn is None
                new_fn = f'type{d}_{img["file_name"]}'
        assert new_fn is not None
        os.rename(src=os.path.join(dataset_dir, 'images', img['file_name']), dst=os.path.join(dataset_dir, 'images', new_fn))
        img['file_name'] = new_fn
    
    print('Removing 0-size boxes')
    print('Before', len(dataset['annotations']))
    dataset['annotations'] = [annot for annot in dataset['annotations'] if annot['bbox'][2] != 0 and annot['bbox'][3] != 0]
    print('After', len(dataset['annotations']))
    utils.save_json(dataset, os.path.join(dataset_dir, f'S-UODAC.{suffix}.json'))
    
dataset = utils.read_all_datasets([os.path.join(dataset_dir, f'S-UODAC.{suffix}.json') for suffix in ['train', 'test']])
utils.save_json(dataset, os.path.join(dataset_dir, 'S-UODAC.json'))
dataset = utils.read_json(os.path.join(dataset_dir, 'S-UODAC.json'))

images: 4669
annotations: 35491
categories: dict_values(['echinus', 'starfish', 'holothurian', 'scallop'])
echinus: 19088
starfish: 5826
holothurian: 4683
scallop: 5894
images: 785
annotations: 5945
categories: dict_values(['echinus', 'starfish', 'holothurian', 'scallop'])
echinus: 3254
starfish: 1015
holothurian: 851
scallop: 825
INFO: using first available "info": {'description': 'Synthetic Underwater Object Detection Algorithm Contest 2020 (https://doi.org/10.1016/j.neucom.2023.01.053)'}
INFO: merged licenses: []
INFO: merged categories: [{'id': 1, 'name': 'echinus'}, {'id': 2, 'name': 'starfish'}, {'id': 3, 'name': 'holothurian'}, {'id': 4, 'name': 'scallop'}]
INFO: merged images: 5454 record(s)
INFO: merged annotations: 41436 record(s)
images: 5454
annotations: 41436
categories: dict_values(['echinus', 'starfish', 'holothurian', 'scallop'])
echinus: 22342
starfish: 6841
holothurian: 5534
scallop: 6719


# Jellytoring

In [2]:
dataset_dir = os.path.join(data_dir, 'resources', 'Jellytoring')
dataset = utils.voc_to_coco(img_dir=os.path.join(dataset_dir, 'data_regions', 'images'), voc_dir=os.path.join(dataset_dir, 'data_regions', 'annotations'), coco_fn=os.path.join(dataset_dir, 'annotations_fixed.json'),
                            info='Jellytoring 2.0 dataset (all regions combined): https://doi.org/10.1002/ece3.9472', fix_voc_filenames=True, fix_cat_names=True, sort_cat_names=True)

images: 2886
annotations: 4636
categories: dict_values(['A. aurita', 'C. achlyos', 'C. branchi', 'C. capilata', 'C. fuscesens', 'C. hysoscella', 'C. lamarckii', 'C. quinquecirrha', 'C. tuberculata', 'N. nemurai', 'P. noctiluca', 'R. lutem', 'R. pulmo', 'S. meleagris', 'T. ohboya'])
A. aurita: 1314
C. achlyos: 339
C. branchi: 151
C. capilata: 202
C. fuscesens: 292
C. hysoscella: 385
C. lamarckii: 124
C. quinquecirrha: 127
C. tuberculata: 353
N. nemurai: 63
P. noctiluca: 498
R. lutem: 108
R. pulmo: 297
S. meleagris: 130
T. ohboya: 253


# GLOW low visibility estuaries

In [81]:
dataset_dir = os.path.join(data_dir, 'resources', 'GLOW_low_visibility_estuaries')
dataset = utils.read_json(os.path.join(dataset_dir, 'raw', 'low_visibility_estuaries.original.json'), assert_correct=False)

In [62]:
print(dataset['annotations'][60258])
print(len(dataset['annotations']))
del dataset['annotations'][60258]
print(len(dataset['annotations']))

old_id = dataset['categories'][0]['id']
assert old_id == 0
new_id = dataset['categories'][-1]['id'] + 1
assert new_id not in [c['id'] for c in dataset['categories']]
dataset['categories'][0]['id'] = new_id
for ann in dataset['annotations']:
    if ann['category_id'] == old_id:
        ann['category_id'] = new_id
utils.sort_categories(dataset)
print(dataset['categories'])

{'originalId': 'JPPfH0amq', 'image_id': 18792, 'category_id': 19, 'bbox': [0, 0, 0, 0], 'className': 'Yellowfin Bream | Tarwhine', 'iscrowd': 0, 'area': 0, 'segmentation': '[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]]', 'id': 60258}
64543
64542
[{'id': 1, 'name': 'Australasian Snapper', 'supercategory': ''}, {'id': 2, 'name': 'Bengal Sergeant', 'supercategory': ''}, {'id': 3, 'name': 'Black-Banded Trevally', 'supercategory': ''}, {'id': 4, 'name': 'Blue Catfish', 'supercategory': ''}, {'id': 5, 'name': 'Blue Swimmer Crab', 'supercategory': ''}, {'id': 6, 'name': 'Eastern Striped Grunter', 'supercategory': ''}, {'id': 7, 'name': 'Eastern Stripey', 'supercategory': ''}, {'id': 8, 'name': 'Echinoderm', 'supercategory': ''}, {'id': 9, 'name': 'Fanbelly Leatherjacket', 'supercategory': ''}, {'id': 10, 'name': 'Gunthers Wrasse', 'supercategory': ''}, {'id': 11, 'name': 'Mackerel sp', 'supercategory': ''}, {'id': 12, 'name': 'Moses Snappe

In [63]:
img_db = {}
for img in dataset['images']:
    img_db[img['file_name']] = img
df = pd.read_csv(os.path.join(dataset_dir, 'raw', 'filename_to_prefix.csv'), header=None)
df.columns = ['file_name', 'prefix']
for idx in df.index:
    prefix = df.loc[idx, 'prefix']
    old_fn = df.loc[idx, 'file_name']
    new_fn = f"{prefix}_{old_fn}"
    img_db[old_fn]['file_name'] = new_fn
    os.rename(src=os.path.join(dataset_dir, 'images', old_fn), dst=os.path.join(dataset_dir, 'images', new_fn))

In [92]:
# drop these because we do not know the location and time of recording and that could confound the analysis
print(len(dataset['images']), len(dataset['annotations']))
dataset['images'] = [img for img in dataset['images'] if not img['file_name'].startswith('Highlights')]
dataset['annotations'] = utils.filter_annotations_with_images(dataset)
print(len(dataset['images']), len(dataset['annotations']))

19557 64542
19041 61991


In [ ]:
for ann in dataset['annotations']:
    if isinstance(ann['segmentation'], str):
        ann['segmentation'] = eval(ann['segmentation'].replace('null', 'None'))
        assert isinstance(ann['segmentation'], list)
        assert isinstance(ann['segmentation'][0], list)
        assert ann['segmentation'][0][0] is None or isinstance(ann['segmentation'][0][0], int)
    else:
        print(ann['id'])

In [93]:
dataset['info']['description'] = 'MBEEC (Moreton Bay Environmental Education Centre) dataset of estuarine fish in poor visibility conditions (https://github.com/slopezmarcano/dataset-fish-detection-low-visibility)'
utils.save_json(dataset, os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.all.json'))

### Subset to target species

In [20]:
dataset_dir = os.path.join(data_dir, 'resources', 'GLOW_low_visibility_estuaries')
dataset = utils.read_json(os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.all.json'))

images: 15549
annotations: 54977
categories: dict_values(['Fish', 'Australasian Snapper', 'Paradise Threadfin Bream', 'Smallmouth Scad', 'Smooth Golden Toadfish'])
Fish: 23840
Australasian Snapper: 8879
Paradise Threadfin Bream: 10321
Smallmouth Scad: 6923
Smooth Golden Toadfish: 5014


In [67]:
dataset['categories'] = [
    {'id': 1, 'name': 'Fish', 'supercategory': ''},
    {'id': 2, 'name': 'Australasian Snapper', 'supercategory': ''},
    {'id': 3, 'name': 'Paradise Threadfin Bream', 'supercategory': ''},
    {'id': 4, 'name': 'Smallmouth Scad', 'supercategory': ''},
    {'id': 5, 'name': 'Smooth Golden Toadfish', 'supercategory': ''},
]

id_map = {
    1: 2,
    2: 1,
    3: 1,
    4: 1,
    5: 1,
    6: 1,
    7: 1,
    8: 1,
    9: 1,
    10: 1,
    11: 1,
    12: 1,
    13: 3,
    14: 1,
    15: 1,
    16: 1,
    17: 4,
    18: 5,
    19: 1,
    20: 1,
}

In [69]:
for ann in dataset['annotations']:
    ann['category_id'] = id_map[ann['category_id']]
    
annotations = utils.get_annotations_dict(dataset)
remove = []
for img_id in annotations:
    if all([ann['category_id'] == 1 for ann in annotations[img_id]]):
        remove.append(img_id)
        
keep = []
for img in dataset['images']:
    if img['id'] not in remove:
        keep.append(img)
dataset['images'] = keep

dataset['annotations'] = utils.filter_annotations_with_images(dataset)

utils.save_json(dataset, os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.target.json'))

### Subset to abundant

In [82]:
dataset_dir = os.path.join(data_dir, 'resources', 'GLOW_low_visibility_estuaries')
dataset = utils.read_json(os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.all.json'))

images: 19041
annotations: 61991
categories: dict_values(['Australasian Snapper', 'Bengal Sergeant', 'Black-Banded Trevally', 'Blue Catfish', 'Blue Swimmer Crab', 'Eastern Striped Grunter', 'Eastern Stripey', 'Echinoderm', 'Fanbelly Leatherjacket', 'Gunthers Wrasse', 'Mackerel sp', 'Moses Snapper', 'Paradise Threadfin Bream', 'Pinkbanded Grubfish', 'Pomacentrid sp', 'Remora sp', 'Smallmouth Scad', 'Smooth Golden Toadfish', 'Unknown1', 'Yellowfin Bream | Tarwhine'])
Australasian Snapper: 8879
Bengal Sergeant: 277
Black-Banded Trevally: 89
Blue Catfish: 2207
Blue Swimmer Crab: 745
Eastern Striped Grunter: 14554
Eastern Stripey: 307
Echinoderm: 14
Fanbelly Leatherjacket: 190
Gunthers Wrasse: 603
Mackerel sp: 139
Moses Snapper: 53
Paradise Threadfin Bream: 10321
Pinkbanded Grubfish: 502
Pomacentrid sp: 27
Remora sp: 41
Smallmouth Scad: 6923
Smooth Golden Toadfish: 5014
Unknown1: 6
Yellowfin Bream | Tarwhine: 11100


In [83]:
dataset['categories'] = [
    {'id': 1, 'name': 'Fish', 'supercategory': ''},
    {'id': 2, 'name': 'Australasian Snapper', 'supercategory': ''},
    {'id': 3, 'name': 'Eastern Striped Grunter', 'supercategory': ''},
    {'id': 4, 'name': 'Paradise Threadfin Bream', 'supercategory': ''},
    {'id': 5, 'name': 'Smallmouth Scad', 'supercategory': ''},
    {'id': 6, 'name': 'Smooth Golden Toadfish', 'supercategory': ''},
    {'id': 7, 'name': 'Yellowfin Bream | Tarwhine', 'supercategory': ''}
]

id_map = {
    1: 2,
    2: 1,
    3: 1,
    4: 1,
    5: 1,
    6: 3,
    7: 1,
    8: 1,
    9: 1,
    10: 1,
    11: 1,
    12: 1,
    13: 4,
    14: 1,
    15: 1,
    16: 1,
    17: 5,
    18: 6,
    19: 1,
    20: 7,
}

In [84]:
for ann in dataset['annotations']:
    ann['category_id'] = id_map[ann['category_id']]
    
annotations = utils.get_annotations_dict(dataset)
remove = []
for img_id in annotations:
    if all([ann['category_id'] == 1 for ann in annotations[img_id]]):
        remove.append(img_id)
        
keep = []
for img in dataset['images']:
    if img['id'] not in remove:
        keep.append(img)
dataset['images'] = keep

dataset['annotations'] = utils.filter_annotations_with_images(dataset)

utils.save_json(dataset, os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.abundant.json'))

images: 16540
annotations: 59001
categories: dict_values(['Fish', 'Australasian Snapper', 'Eastern Striped Grunter', 'Paradise Threadfin Bream', 'Smallmouth Scad', 'Smooth Golden Toadfish', 'Yellowfin Bream | Tarwhine'])
Fish: 2210
Australasian Snapper: 8879
Eastern Striped Grunter: 14554
Paradise Threadfin Bream: 10321
Smallmouth Scad: 6923
Smooth Golden Toadfish: 5014
Yellowfin Bream | Tarwhine: 11100


In [27]:
dataset_dir = os.path.join(data_dir, 'resources', 'GLOW_low_visibility_estuaries')
dataset = utils.read_json(os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.abundant.json'))
random_state = np.random.RandomState(0)

imgs = np.asarray([img['file_name'] for img in dataset['images']])
idx = np.argsort(imgs)

prev_frame = None
frames = []
select = []
for i in idx:
    s = imgs[i].split('.')
    assert s[-1] == 'jpg'
    assert s[-2].lower().startswith('mp4_5fps_') or s[-2].lower().startswith('mp4_7fps_') or s[-2].lower().startswith('mp4_25fps_')
    frame = int(s[-2].split('_')[-1])
    fps = int(s[-2].split('_')[-2][:-3])
    prefix = '.'.join(s[:-2])
    
    if prev_frame is None or (prefix == prev_prefix and frame - prev_frame == 1):
        assert prev_frame is None or fps == prev_fps
        frames.append(i)
        prev_frame = frame
        prev_fps = fps
        prev_prefix = prefix
    else:
        # print(prev_prefix, prev_fps)
        # print(frames)
        selected = frames[::prev_fps] if len(frames) > prev_fps else random_state.choice(frames, 1)
        # print(selected)
        # print('')
        select.extend(selected)
        frames = []
        prev_frame = None
        prev_prefix = None

dataset['images'] = np.asarray(dataset['images'])[select].tolist()
dataset['annotations'] = utils.filter_annotations_with_images(dataset)
fish_cat_id = [cat['id'] for cat in dataset['categories'] if cat['name'] == 'Fish']
assert len(fish_cat_id) == 1
utils.remove_category(dataset=dataset, category_id=fish_cat_id[0], verbose=False)
utils.save_json(dataset, os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.abundant.1fps.json'))

print('\nAfter subsampling FPS\n')
dataset = utils.read_json(os.path.join(dataset_dir, 'GLOW_low_visibility_estuaries.abundant.1fps.json'))

images: 16540
annotations: 59001
categories: dict_values(['Fish', 'Australasian Snapper', 'Eastern Striped Grunter', 'Paradise Threadfin Bream', 'Smallmouth Scad', 'Smooth Golden Toadfish', 'Yellowfin Bream | Tarwhine'])
Fish: 2210
Australasian Snapper: 8879
Eastern Striped Grunter: 14554
Paradise Threadfin Bream: 10321
Smallmouth Scad: 6923
Smooth Golden Toadfish: 5014
Yellowfin Bream | Tarwhine: 11100

After subsampling FPS

images: 3327
annotations: 10377
categories: dict_values(['Australasian Snapper', 'Eastern Striped Grunter', 'Paradise Threadfin Bream', 'Smallmouth Scad', 'Smooth Golden Toadfish', 'Yellowfin Bream | Tarwhine'])
Australasian Snapper: 1754
Eastern Striped Grunter: 1992
Paradise Threadfin Bream: 2013
Smallmouth Scad: 1428
Smooth Golden Toadfish: 1023
Yellowfin Bream | Tarwhine: 2167


# Statistics for all datasets

In [ ]:
for d in ['DeepFish', 'Jellytoring', 'GLOW_low_visibility_estuaries', 'S-UODAC']:
    dataset = utils.read_json(os.path.join(data_dir, 'resources', d, f'{d}.json'), verbose=False)
    print(d)
    print('images', len(dataset['images']))
    print('annotations', len(dataset['annotations']))
    print('annot per img', len(dataset['annotations'])/len(dataset['images']))
    print('categories', len(dataset['categories']))
    print([c['name'] for c in dataset['categories']])
    print(len(set(utils.get_filename_groups(dataset['images']))))
    print('')
    # img_to_anns = utils.get_annotations_dict(dataset)
    # print(d, np.max([len(img_to_anns[img]) for img in img_to_anns]))